In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from itertools import combinations
import numpy as np
from xgboost import XGBClassifier

In [ ]:
data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\Cancer Prediction v2. - DS1 (1).csv" 
df = pd.read_csv(data_path, delimiter=",")

print(df)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# print(np.bincount(y))
smote = SMOTE(random_state=42, k_neighbors=5)
# X, y = smote.fit_resample(X, y)
# print(np.bincount(y))

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_cv, y_train, y_cv = train_test_split(X_temp, y_temp, test_size=1/8, random_state=42, stratify=y_temp)

print(X_train.shape)
print(X_cv.shape)
print(X_test.shape)

     200717_x_at  202192_s_at  203592_s_at  207574_s_at  209304_x_at  \
0      -0.013204     0.143578     0.363944    -0.217254     0.113607   
1       0.776570    -0.189628    -0.650692    -0.508355    -0.501216   
2      -1.183713    -0.625632     0.495192    -0.306498    -0.794870   
3       0.984420    -0.399410    -0.806296    -0.773497    -1.017541   
4       0.746004    -0.779731    -1.619350    -0.286077    -0.708715   
..           ...          ...          ...          ...          ...   
304     1.578388     0.149185    -0.287055     2.202842     2.375767   
305    -0.107202     1.306598     0.238374     0.806519     0.838113   
306     1.846280    -0.008404    -0.524969    -0.592765    -0.387908   
307     0.791526     0.319873    -0.239817    -0.231235    -0.034546   
308    -0.060069     0.173836    -0.064435     0.363354     0.123174   

     212356_at  218686_s_at  219173_at  37796_at  characteristics_ch1.14  
0     0.599693    -1.596879  -0.474819 -0.237991            

In [21]:
external_test_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\ET1 (1).csv"
df_external = pd.read_csv(external_test_path, delimiter="\t")
X_external = df_external.iloc[:, :-1]
y_external = df_external.iloc[:, -1]

print(X_external)
print(y_external)

     200717_x_at  202192_s_at  203592_s_at  207574_s_at  209304_x_at  \
0      -0.135788     0.939685     0.430941     0.136428     0.033315   
1       0.519233     2.165193     0.485440     0.902022     1.040215   
2      -3.233226     1.298075     0.669990    -0.250928    -0.758057   
3      -1.387636     1.115187     2.246421     1.320521     2.418789   
4      -3.567574     3.138366     0.545791     1.350926     1.265927   
..           ...          ...          ...          ...          ...   
324     0.722058    -0.367874    -1.827396    -2.426716    -2.410160   
325    -1.320700    -0.824948    -0.088030    -0.541048    -0.960907   
326     0.464549    -0.580506    -0.128712     0.151156     0.225694   
327     0.008931    -1.742937     0.239725     0.291903     0.144358   
328    -1.872695    -0.388428    -0.357444    -1.311074    -1.328603   

     212356_at  218686_s_at  219173_at  37796_at  
0     0.435264     1.017310   1.324561  0.794945  
1     1.901970     1.353316   1.3

In [25]:
from sklearn.feature_selection import SelectKBest, f_classif, RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

models = {
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True, kernel='rbf', C=1.0, random_state=42)),
}

filter_k = 9

for name, model in models.items():
    print(f"\n==== Model: {name} ====")

    filter_selector = SelectKBest(score_func=f_classif, k=filter_k)
    wrapper_selector = RFECV(
        estimator=LogisticRegression(solver='liblinear'),
        step=1,
        cv=StratifiedKFold(5),
        scoring='roc_auc',
        n_jobs=-1
    )

    feature_pipeline = Pipeline([
        ('filter', filter_selector),
        ('wrapper', wrapper_selector)
    ])

    X_train_selected = feature_pipeline.fit_transform(X_train, y_train)
    X_val_selected = feature_pipeline.transform(X_cv)
    X_test_selected = feature_pipeline.transform(X_test)
    X_external_selected = feature_pipeline.transform(X_external)

    model.fit(X_train_selected, y_train)

    for split_name, X_set, y_set in zip(
        ['Train', 'Validation', 'Internal Test', 'External Test'],
        [X_train_selected, X_val_selected, X_test_selected, X_external_selected],
        [y_train, y_cv, y_test, y_external]
    ):
        probas = model.predict_proba(X_set)[:, 1]
        preds = model.predict(X_set)

        auc = roc_auc_score(y_set, probas)
        acc = accuracy_score(y_set, preds)

        print(f"{split_name} - AUC: {auc:.4f} | Accuracy: {acc:.4f}")



==== Model: SVM ====
Train - AUC: 0.9212 | Accuracy: 0.7593
Validation - AUC: 0.5707 | Accuracy: 0.7419
Internal Test - AUC: 0.7106 | Accuracy: 0.7581
External Test - AUC: 0.6511 | Accuracy: 0.8906
